# Generation de Chansons Completes : YuE vs SongGeneration 2

**Module :** 02-Audio-Advanced  
**Niveau :** Avance  
**Technologies :** YuE (M-A-P), SongGeneration 2 / LeVo (Tencent), 10-24 GB VRAM  
**Duree estimee :** 60 minutes  

## Objectifs d'Apprentissage

- [ ] Comprendre la difference entre text-to-music (MusicGen) et text-to-song (voix + accompagnement)
- [ ] Comparer les architectures YuE (autoregressive 2 stages) et SongGeneration 2 (LLM-Diffusion hybride)
- [ ] Installer et configurer les deux modeles pour un GPU 24 GB
- [ ] Generer des chansons avec paroles a partir de descriptions de style
- [ ] Evaluer les resultats sur la fidelite des paroles, la qualite vocale et la musicalite
- [ ] Separer les pistes vocales et instrumentales
- [ ] Identifier les cas d'usage adaptes a chaque modele

## Prerequis

- GPU NVIDIA avec 10+ GB VRAM (base) ou 24 GB (modeles complets)
- `pip install torch torchaudio transformers accelerate einops sentencepiece tqdm`
- FlashAttention 2 recommande pour YuE sur 24 GB
- Git LFS pour le telechargement des modeles
- Connaissance des concepts TTS et generation musicale (notebooks 02-1 a 02-6)

**Navigation :** [<< 02-6](02-6-MIDI-Generation.ipynb) | [Index](../README.md) | [03-1 >>](../03-Orchestration/03-1-Multi-Model-Audio-Comparison.ipynb)

In [ ]:
# Parametres Papermill - JAMAIS modifier ce commentaire

# Configuration notebook
notebook_mode = "interactive"        # "interactive" ou "batch"
skip_widgets = False               # True pour mode batch MCP
debug_level = "INFO"

# Choix des modeles a tester
test_yue = True                    # Tester YuE
test_songgen = True                # Tester SongGeneration 2

# Configuration YuE
yue_stage1_model = "m-a-p/YuE-s1-7B-anneal-en-cot"  # Stage 1 (7B)
yue_stage2_model = "m-a-p/YuE-s2-1B-general"        # Stage 2 (1B)
yue_n_segments = 2                 # Nombre de segments (~30s chacun)
yue_max_tokens = 3000              # Tokens par segment

# Configuration SongGeneration 2
songgen_model = "lglg666/SongGeneration-base-new"    # base-new pour 10 GB
songgen_separate = True            # Pistes separees vocal/bgm

# Configuration generale
device = "cuda"                    # "cuda" ou "cpu"
save_results = True                # Sauvegarder les fichiers
compare_models = True              # Generer le tableau comparatif

Les parametres Papermill configurent les modeles a tester (YuE et SongGeneration 2), le device GPU/CPU et les options de generation. La cellule suivante initialise l'environnement Python et verifie la disponibilite du GPU.

In [2]:
# Setup environnement et imports
import os
import sys
import time
import gc
import json
import subprocess
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Any, Optional
import logging

import numpy as np
from IPython.display import Audio, display, HTML

# Import helpers GenAI
GENAI_ROOT = Path.cwd()
while GENAI_ROOT.name != 'GenAI' and len(GENAI_ROOT.parts) > 1:
    GENAI_ROOT = GENAI_ROOT.parent

HELPERS_PATH = GENAI_ROOT / 'shared' / 'helpers'
if HELPERS_PATH.exists():
    sys.path.insert(0, str(HELPERS_PATH.parent))
    try:
        from helpers.audio_helpers import play_audio, save_audio
        print("Helpers audio importes")
    except ImportError:
        print("Helpers audio non disponibles - mode autonome")

# Repertoires
OUTPUT_DIR = GENAI_ROOT / 'outputs' / 'audio' / 'songs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODELS_DIR = GENAI_ROOT / 'models' / 'song-generation'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Configuration logging
logging.basicConfig(level=getattr(logging, debug_level))
logger = logging.getLogger('song-generation')

# Verification GPU
gpu_available = False
gpu_vram = 0
try:
    import torch
    gpu_available = torch.cuda.is_available()
    if gpu_available:
        gpu_name = torch.cuda.get_device_name(0)
        gpu_vram = torch.cuda.get_device_properties(0).total_mem / (1024**3)
        print(f"GPU : {gpu_name} ({gpu_vram:.1f} GB VRAM)")
    else:
        print("GPU non disponible - generation de chansons requiert un GPU")
        if device == "cuda":
            device = "cpu"
except ImportError:
    print("torch non installe")
    device = "cpu"

print(f"\nGeneration de Chansons Completes")
print(f"Date : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Mode : {notebook_mode}, Device : {device}")
print(f"YuE : {'actif' if test_yue else 'desactive'}")
print(f"SongGeneration 2 : {'actif' if test_songgen else 'desactive'}")
print(f"Sortie : {OUTPUT_DIR}")

Helpers audio importes


GPU non disponible - generation de chansons requiert un GPU

Generation de Chansons Completes
Date : 2026-03-22 13:35:04
Mode : interactive, Device : cpu
YuE : actif
SongGeneration 2 : actif
Sortie : MyIA.AI.Notebooks\GenAI\outputs\audio\songs


L'environnement Python est pret et les repertoires de sortie sont crees. La cellule suivante charge le fichier `.env` pour recuperer le token HuggingFace requis pour telecharger les poids de YuE (7B+1B) et SongGeneration 2.

In [3]:
# Chargement robuste de la configuration .env
from dotenv import load_dotenv
import os

current_path = Path.cwd()
env_loaded = False
for _ in range(10):
    env_path = current_path / ".env"
    if env_path.exists():
        load_dotenv(env_path)
        print(f".env charge depuis: {env_path}")
        env_loaded = True
        break
    current_path = current_path.parent
    if current_path.name == "GenAI" or len(current_path.parts) <= 1:
        break
if not env_loaded:
    print("WARNING: .env non trouve, utilisation variables environnement")

hf_token = os.environ.get("HUGGINGFACE_TOKEN") or os.environ.get("HF_TOKEN")
if hf_token:
    print("Token HuggingFace disponible")
else:
    print("Token HuggingFace non disponible (modeles publics uniquement)")

Token HuggingFace non disponible (modeles publics uniquement)


## Dependances GPU Optionnelles

Ce notebook requiert un GPU pour la generation. Sans GPU, seule l'analyse architecturale est disponible.

- **SongGeneration 2 base** : 10 GB VRAM minimum
- **YuE (bf16 + FlashAttn)** : 24 GB VRAM
- **YuE quantise (exllamav2)** : 6 GB VRAM

## Section 1 : Text-to-Song vs Text-to-Music

La generation de chansons completes (text-to-song) est fondamentalement differente de la generation musicale (text-to-music) vue dans les notebooks precedents :

| Aspect | Text-to-Music (MusicGen) | Text-to-Song (YuE, SongGen2) |
|--------|--------------------------|------------------------------|
| **Sortie** | Musique instrumentale | Voix chantee + accompagnement |
| **Entree texte** | Description de style | Paroles structurees (couplet, refrain) |
| **Alignement** | Style global | Mot par mot (lyrics alignment) |
| **Structure** | Forme libre | Couplet-refrain-pont |
| **Complexite** | Moderate (300M-3.3B) | Elevee (4B-8B) |
| **VRAM** | 4-20 GB | 10-24 GB |

### Panorama des modeles text-to-song (2025)

| Modele | Equipe | Params | Open-source | Qualite |
|--------|--------|--------|-------------|--------|
| **YuE** | HKUST / M-A-P | 7B+1B | Apache 2.0 | Bonne |
| **SongGeneration 2** | Tencent AI Lab | 4B | Custom | Excellente |
| Suno v5 | Suno | ? | Non | Reference commerciale |
| Udio | Udio | ? | Non | Reference commerciale |
| ACE-Song | ACE Studio | ? | Non | Commerciale |

YuE et SongGeneration 2 sont les deux principaux modeles open-source. Ce notebook les compare en pratique.

## Section 2 : Architectures comparees

### YuE : Autoregressive 2 Stages

| Stage | Modele | Role | Parametres |
|-------|--------|------|------------|
| Stage 1 | LLaMA 7B | Genere des tokens audio grossiers a partir des paroles | 7B |
| Stage 2 | LLaMA 1B | Raffine les tokens en audio haute fidelite | 1B |
| Vocodeur | xcodec_mini | Convertit les tokens en forme d'onde | - |

```
Paroles + Genre --> [Stage 1, 7B] --> tokens grossiers
                                          |
                                    [Stage 2, 1B] --> tokens raffines
                                          |
                                    [Vocodeur] --> audio 44.1kHz
```

**Modes** :
- **CoT (Chain-of-Thought)** : Le modele raisonne sur la structure musicale avant de generer
- **ICL (In-Context Learning)** : Style transfer a partir d'un audio de reference (~30s)

### SongGeneration 2 / LeVo 2 : LLM-Diffusion hybride

| Composant | Role | Parametres |
|-----------|------|------------|
| LeLM (Language Model) | Planifie la structure, melodie et harmonie | ~2B |
| Diffusion Module | Rend l'audio haute fidelite | ~2B |
| Codec | Encodage/decodage audio | - |

```
Paroles + Description --> [LeLM] --> plan musical structure
                                          |
                                    [Diffusion] --> audio 48kHz FLAC
```

### Comparaison architecturale

| Aspect | YuE | SongGeneration 2 |
|--------|-----|-------------------|
| Approche | Autoregressive pur | Hybride LLM + Diffusion |
| Avantage | Coherence longue portee | Qualite audio superieure |
| Inconvenient | Lent (token par token) | Moins flexible en longueur |
| Sortie | 44.1kHz MP3 | 48kHz FLAC |
| Vitesse (30s) | ~6 min (RTX 4090) | ~25s (RTF 0.82) |
| Duree max | Illimitee (avec VRAM) | 4m30 |

## Section 3 : Installation et configuration

Les deux modeles necessitent le clonage de leurs repositories et le telechargement de checkpoints volumineux. Cette section verifie les prerequis et prepare l'environnement.

In [4]:
# Verification des prerequis et installation
print("VERIFICATION DES PREREQUIS")
print("=" * 45)

# Verifier les dependances communes
deps_status = {}
for pkg in ["torch", "torchaudio", "transformers", "accelerate",
            "einops", "sentencepiece", "tqdm", "soundfile"]:
    try:
        __import__(pkg)
        deps_status[pkg] = True
    except ImportError:
        deps_status[pkg] = False

missing = [k for k, v in deps_status.items() if not v]
installed = [k for k, v in deps_status.items() if v]

print(f"Installes : {', '.join(installed)}")
if missing:
    print(f"Manquants : {', '.join(missing)}")
    print(f"  pip install {' '.join(missing)}")
else:
    print("Toutes les dependances communes sont installees")

# Verifier FlashAttention (important pour YuE)
flash_attn_available = False
try:
    import flash_attn
    flash_attn_available = True
    print(f"FlashAttention {flash_attn.__version__} disponible")
except ImportError:
    print("FlashAttention non installe (recommande pour YuE sur 24 GB)")

# Verifier git-lfs
try:
    result = subprocess.run(["git", "lfs", "version"], capture_output=True, text=True)
    if result.returncode == 0:
        print(f"Git LFS : {result.stdout.strip()}")
    else:
        print("Git LFS non installe (requis pour telechargement modeles)")
except FileNotFoundError:
    print("Git non disponible")

# Resume VRAM
print(f"\nRecommandations VRAM :")
if gpu_available:
    print(f"  GPU disponible : {gpu_vram:.0f} GB")
    if gpu_vram >= 24:
        print(f"  YuE (bf16) : OK")
        print(f"  SongGeneration 2 (v2-large) : OK (sans prompt audio)")
    elif gpu_vram >= 10:
        print(f"  YuE (bf16) : NON (24 GB requis, utiliser quantize)")
        print(f"  SongGeneration 2 (base) : OK")
    else:
        print(f"  VRAM insuffisante pour la generation de chansons")
else:
    print(f"  Pas de GPU - demonstration architecturale uniquement")

VERIFICATION DES PREREQUIS


Installes : torch, transformers, accelerate, tqdm, soundfile
Manquants : torchaudio, einops, sentencepiece
  pip install torchaudio einops sentencepiece
FlashAttention non installe (recommande pour YuE sur 24 GB)


Git LFS : git-lfs/3.6.1 (GitHub; windows amd64; go 1.23.3; git ea47a34b)

Recommandations VRAM :
  Pas de GPU - demonstration architecturale uniquement


Les prerequis Python sont verifies et les instructions d'installation sont affichees. La cellule suivante configure les chemins d'acces aux depots YuE et SongGeneration clones localement.

In [5]:
# Configuration des repositories et modeles
print("CONFIGURATION DES MODELES")
print("=" * 45)

YUE_DIR = MODELS_DIR / "YuE"
SONGGEN_DIR = MODELS_DIR / "SongGeneration"

# Verifier si YuE est installe
yue_ready = False
if test_yue:
    yue_infer = YUE_DIR / "inference" / "infer.py"
    if yue_infer.exists():
        print(f"YuE : installe dans {YUE_DIR}")
        yue_ready = True
    else:
        print(f"YuE : non installe")
        print(f"  Pour installer :")
        print(f"  cd {MODELS_DIR}")
        print(f"  git clone https://github.com/multimodal-art-projection/YuE.git")
        print(f"  cd YuE/inference && git clone https://huggingface.co/m-a-p/xcodec_mini_infer")
        print(f"  pip install flash-attn --no-build-isolation  # Pour 24 GB")

# Verifier si SongGeneration 2 est installe
songgen_ready = False
if test_songgen:
    songgen_script = SONGGEN_DIR / "generate.sh"
    if songgen_script.exists():
        print(f"SongGeneration 2 : installe dans {SONGGEN_DIR}")
        songgen_ready = True
    else:
        print(f"SongGeneration 2 : non installe")
        print(f"  Pour installer :")
        print(f"  cd {MODELS_DIR}")
        print(f"  git clone https://github.com/tencent-ailab/SongGeneration.git")
        print(f"  cd SongGeneration")
        print(f"  pip install -r requirements.txt")
        print(f"  huggingface-cli download lglg666/SongGeneration-Runtime --local-dir ./runtime")
        print(f"  mv runtime/ckpt ckpt && mv runtime/third_party third_party")
        print(f"  huggingface-cli download {songgen_model} --local-dir ./songgeneration_model")

if not yue_ready and not songgen_ready:
    print(f"\nAucun modele installe - le notebook continuera en mode demonstration")

CONFIGURATION DES MODELES
YuE : non installe
  Pour installer :
  cd MyIA.AI.Notebooks\GenAI\models\song-generation
  git clone https://github.com/multimodal-art-projection/YuE.git
  cd YuE/inference && git clone https://huggingface.co/m-a-p/xcodec_mini_infer
  pip install flash-attn --no-build-isolation  # Pour 24 GB
SongGeneration 2 : non installe
  Pour installer :
  cd MyIA.AI.Notebooks\GenAI\models\song-generation
  git clone https://github.com/tencent-ailab/SongGeneration.git
  cd SongGeneration
  pip install -r requirements.txt
  huggingface-cli download lglg666/SongGeneration-Runtime --local-dir ./runtime
  mv runtime/ckpt ckpt && mv runtime/third_party third_party
  huggingface-cli download lglg666/SongGeneration-base-new --local-dir ./songgeneration_model

Aucun modele installe - le notebook continuera en mode demonstration


### Interpretation : Installation

| Modele | Taille telechargement | Temps installation | Difficulte |
|--------|----------------------|-------------------|------------|
| YuE (7B+1B) | ~16 GB | 15-30 min | Moyenne (FlashAttn requis) |
| SongGen 2 base | ~4 GB + runtime | 10-20 min | Facile |
| SongGen 2 v2-large | ~10 GB + runtime | 15-30 min | Facile |

**Points cles** :
1. Les deux modeles necessitent le clonage de repositories entiers (pas de simple `pip install`)
2. FlashAttention 2 est critique pour YuE sur 24 GB (OOM sans)
3. SongGeneration 2 est plus simple a installer et plus leger

## Section 4 : Preparation des paroles

Les deux modeles necessitent des paroles structurees avec des marqueurs de section. Le formatage est crucial pour la qualite du resultat.

### Format YuE

```
[verse]
Walking through the morning light
Coffee steam and city sights

[chorus]
Sing it loud, sing it free
This is where I want to be
```

### Format SongGeneration 2

```
[intro-short] ; [verse] Walking through the morning light. Coffee steam and city sights. ; [chorus] Sing it loud sing it free. This is where I want to be. ; [outro-short]
```

| Element | YuE | SongGeneration 2 |
|---------|-----|-------------------|
| Separateur sections | Ligne vide | ` ; ` (point-virgule) |
| Separateur phrases | Retour a la ligne | `. ` (point) |
| Marqueurs | `[verse]`, `[chorus]`, `[bridge]` | + `[intro-short/medium]`, `[outro-short/medium]`, `[inst-short/medium]` |
| Langues | EN, ZH, JA, KO | EN, ZH, ES, JA |

In [6]:
# Preparation des paroles de test
print("PREPARATION DES PAROLES")
print("=" * 45)

# Chanson de test commune
test_song = {
    "title": "Morning Light",
    "genre": "pop, uplifting, piano, acoustic guitar, female vocal",
    "sections": [
        ("verse", [
            "Walking through the morning light",
            "Coffee steam and city sights",
            "Every step a brand new start",
            "Rhythm beating in my heart",
        ]),
        ("chorus", [
            "Sing it loud sing it free",
            "This is where I want to be",
            "Every note a part of me",
            "Music sets my spirit free",
        ]),
    ]
}


def format_lyrics_yue(song):
    """Formate les paroles pour YuE."""
    lines = []
    for section, lyrics in song["sections"]:
        lines.append(f"[{section}]")
        for line in lyrics:
            lines.append(line)
        lines.append("")  # ligne vide entre sections
    return "\n".join(lines).strip()


def format_lyrics_songgen(song):
    """Formate les paroles pour SongGeneration 2."""
    parts = ["[intro-short]"]
    for section, lyrics in song["sections"]:
        text = ". ".join(lyrics) + "."
        parts.append(f"[{section}] {text}")
    parts.append("[outro-short]")
    return " ; ".join(parts)


# Afficher les formats
yue_lyrics = format_lyrics_yue(test_song)
songgen_lyrics = format_lyrics_songgen(test_song)

print(f"Chanson : {test_song['title']}")
print(f"Genre : {test_song['genre']}")
print(f"\n--- Format YuE ---")
print(yue_lyrics)
print(f"\n--- Format SongGeneration 2 ---")
print(songgen_lyrics)

# Sauvegarder les fichiers de paroles
if save_results:
    lyrics_dir = OUTPUT_DIR / "lyrics"
    lyrics_dir.mkdir(parents=True, exist_ok=True)

    with open(str(lyrics_dir / "lyrics_yue.txt"), "w") as f:
        f.write(yue_lyrics)
    with open(str(lyrics_dir / "genre_yue.txt"), "w") as f:
        f.write(test_song["genre"])

    songgen_input = {
        "idx": "morning_light",
        "gt_lyric": songgen_lyrics,
        "descriptions": test_song["genre"],
    }
    with open(str(lyrics_dir / "input_songgen.jsonl"), "w") as f:
        f.write(json.dumps(songgen_input, ensure_ascii=False) + "\n")

    print(f"\nFichiers sauvegardes dans {lyrics_dir}")

PREPARATION DES PAROLES
Chanson : Morning Light
Genre : pop, uplifting, piano, acoustic guitar, female vocal

--- Format YuE ---
[verse]
Walking through the morning light
Coffee steam and city sights
Every step a brand new start
Rhythm beating in my heart

[chorus]
Sing it loud sing it free
This is where I want to be
Every note a part of me
Music sets my spirit free

--- Format SongGeneration 2 ---
[intro-short] ; [verse] Walking through the morning light. Coffee steam and city sights. Every step a brand new start. Rhythm beating in my heart. ; [chorus] Sing it loud sing it free. This is where I want to be. Every note a part of me. Music sets my spirit free. ; [outro-short]

Fichiers sauvegardes dans MyIA.AI.Notebooks\GenAI\outputs\audio\songs\lyrics


## Section 5 : Generation avec YuE

YuE genere des chansons en deux etapes. Le stage 1 (7B) comprend les paroles et planifie la musique. Le stage 2 (1B) affine la qualite audio.

| Parametre | Valeur | Description |
|-----------|--------|-------------|
| `--run_n_segments` | 2 | Segments a generer (1 segment ~ 30s) |
| `--max_new_tokens` | 3000 | Tokens par segment (~30s audio) |
| `--stage2_batch_size` | 4 | Batch size du stage 2 |
| `--repetition_penalty` | 1.1 | Penalite de repetition (1.0-2.0) |
| `--seed` | 42 | Graine aleatoire |

In [7]:
# Generation avec YuE
print("GENERATION AVEC YUE")
print("=" * 45)

yue_result = None

if test_yue and yue_ready and gpu_available:
    print(f"Modele Stage 1 : {yue_stage1_model}")
    print(f"Modele Stage 2 : {yue_stage2_model}")
    print(f"Segments : {yue_n_segments} (~{yue_n_segments * 30}s)")

    # Sauvegarder les fichiers d'entree
    lyrics_path = OUTPUT_DIR / "lyrics" / "lyrics_yue.txt"
    genre_path = OUTPUT_DIR / "lyrics" / "genre_yue.txt"
    yue_output_dir = OUTPUT_DIR / "yue"
    yue_output_dir.mkdir(parents=True, exist_ok=True)

    # Lancer la generation
    print(f"\nGeneration en cours (peut prendre plusieurs minutes)...")
    start_time = time.time()

    cmd = [
        sys.executable, str(YUE_DIR / "inference" / "infer.py"),
        "--cuda_idx", "0",
        "--stage1_model", yue_stage1_model,
        "--stage2_model", yue_stage2_model,
        "--genre_txt", str(genre_path),
        "--lyrics_txt", str(lyrics_path),
        "--run_n_segments", str(yue_n_segments),
        "--stage2_batch_size", "4",
        "--output_dir", str(yue_output_dir),
        "--max_new_tokens", str(yue_max_tokens),
        "--repetition_penalty", "1.1",
        "--seed", "42",
    ]

    try:
        result = subprocess.run(
            cmd, capture_output=True, text=True,
            timeout=600, cwd=str(YUE_DIR / "inference")
        )
        gen_time = time.time() - start_time

        if result.returncode == 0:
            print(f"Generation terminee en {gen_time:.1f}s")

            # Chercher le fichier de sortie
            output_files = list(yue_output_dir.rglob("*.mp3")) + list(yue_output_dir.rglob("*.wav"))
            if output_files:
                output_file = output_files[0]
                print(f"Fichier genere : {output_file.name}")

                import torchaudio
                waveform, sr = torchaudio.load(str(output_file))
                duration = waveform.shape[1] / sr

                yue_result = {
                    "file": output_file,
                    "duration": duration,
                    "gen_time": gen_time,
                    "sample_rate": sr,
                    "rtf": gen_time / duration if duration > 0 else float('inf'),
                }

                print(f"Duree : {duration:.1f}s | SR : {sr} Hz")
                print(f"RTF : {yue_result['rtf']:.2f}x")
                display(Audio(data=waveform.numpy(), rate=sr))
            else:
                print("Aucun fichier audio genere")
        else:
            print(f"Erreur de generation :")
            stderr_lines = result.stderr.strip().split("\n")[-5:]
            for line in stderr_lines:
                print(f"  {line}")
    except subprocess.TimeoutExpired:
        print("Timeout (>10 min) - generation interrompue")
    except Exception as e:
        print(f"Erreur : {type(e).__name__} - {str(e)[:150]}")

elif test_yue and not yue_ready:
    print("YuE non installe - voir Section 3 pour les instructions")
    print(f"\nExemple de commande CLI :")
    print(f"  python infer.py \\")
    print(f"    --stage1_model {yue_stage1_model} \\")
    print(f"    --stage2_model {yue_stage2_model} \\")
    print(f"    --genre_txt genre.txt --lyrics_txt lyrics.txt \\")
    print(f"    --run_n_segments 2 --seed 42")
elif not test_yue:
    print("Test YuE desactive")
else:
    print("GPU non disponible")

GENERATION AVEC YUE
YuE non installe - voir Section 3 pour les instructions

Exemple de commande CLI :
  python infer.py \
    --stage1_model m-a-p/YuE-s1-7B-anneal-en-cot \
    --stage2_model m-a-p/YuE-s2-1B-general \
    --genre_txt genre.txt --lyrics_txt lyrics.txt \
    --run_n_segments 2 --seed 42


### Interpretation : YuE

| Aspect | Observation typique |
|--------|--------------------|
| Qualite vocale | Reconnaissable, parfois metallique |
| Fidelite paroles | Bonne pour l'anglais |
| Musicalite | Variee, coherente sur 2 segments |
| Temps | 5-10 min pour ~60s sur RTX 3090/4090 |

**Points cles** :
1. Le mode CoT (Chain-of-Thought) produit des resultats plus structures que ICL
2. Commencer par `[verse]` ou `[chorus]`, jamais `[intro]` (instable)
3. Limiter le nombre de mots par segment (~30s max par segment)
4. FlashAttention 2 est obligatoire pour 24 GB

## Section 6 : Generation avec SongGeneration 2

SongGeneration 2 utilise une approche hybride LLM-Diffusion qui produit des resultats plus rapides et de meilleure fidelite audio.

| Parametre | Description |
|-----------|-------------|
| `gt_lyric` | Paroles structurees (sections separees par ` ; `) |
| `descriptions` | Tags de style (genre, instruments, voix) |
| `--separate` | Generer pistes vocale et instrumentale separees |
| `--low_mem` | Mode economie memoire (CPU offloading) |
| `auto_prompt_audio_type` | Style auto : Pop, Rock, Jazz, Electronic, etc. |

In [8]:
# Generation avec SongGeneration 2
print("GENERATION AVEC SONGGENERATION 2")
print("=" * 45)

songgen_result = None

if test_songgen and songgen_ready and gpu_available:
    print(f"Modele : {songgen_model}")
    print(f"Pistes separees : {'Oui' if songgen_separate else 'Non'}")

    # Fichier d'entree
    input_jsonl = OUTPUT_DIR / "lyrics" / "input_songgen.jsonl"
    songgen_output_dir = OUTPUT_DIR / "songgen"
    songgen_output_dir.mkdir(parents=True, exist_ok=True)

    print(f"\nGeneration en cours...")
    start_time = time.time()

    # Construire la commande
    cmd = [
        "bash", str(SONGGEN_DIR / "generate.sh"),
        str(SONGGEN_DIR / "songgeneration_model"),
        str(input_jsonl),
        str(songgen_output_dir),
    ]
    if songgen_separate:
        cmd.append("--separate")
    if gpu_vram < 24:
        cmd.append("--low_mem")

    try:
        result = subprocess.run(
            cmd, capture_output=True, text=True,
            timeout=300, cwd=str(SONGGEN_DIR)
        )
        gen_time = time.time() - start_time

        if result.returncode == 0:
            print(f"Generation terminee en {gen_time:.1f}s")

            # Chercher les fichiers de sortie
            output_files = list(songgen_output_dir.rglob("*.flac")) + \
                           list(songgen_output_dir.rglob("*.wav"))
            if output_files:
                # Fichier principal (mix)
                mix_files = [f for f in output_files if "_vocal" not in f.name and "_bgm" not in f.name]
                main_file = mix_files[0] if mix_files else output_files[0]

                import torchaudio
                waveform, sr = torchaudio.load(str(main_file))
                duration = waveform.shape[1] / sr

                songgen_result = {
                    "file": main_file,
                    "duration": duration,
                    "gen_time": gen_time,
                    "sample_rate": sr,
                    "rtf": gen_time / duration if duration > 0 else float('inf'),
                    "all_files": output_files,
                }

                print(f"\nMix : {main_file.name}")
                print(f"Duree : {duration:.1f}s | SR : {sr} Hz")
                print(f"RTF : {songgen_result['rtf']:.2f}x")
                display(Audio(data=waveform.numpy(), rate=sr))

                # Pistes separees
                if songgen_separate:
                    vocal_files = [f for f in output_files if "_vocal" in f.name]
                    bgm_files = [f for f in output_files if "_bgm" in f.name]

                    if vocal_files:
                        print(f"\nPiste vocale :")
                        v_wav, v_sr = torchaudio.load(str(vocal_files[0]))
                        display(Audio(data=v_wav.numpy(), rate=v_sr))

                    if bgm_files:
                        print(f"Accompagnement :")
                        b_wav, b_sr = torchaudio.load(str(bgm_files[0]))
                        display(Audio(data=b_wav.numpy(), rate=b_sr))
            else:
                print("Aucun fichier audio genere")
        else:
            print(f"Erreur de generation :")
            stderr_lines = result.stderr.strip().split("\n")[-5:]
            for line in stderr_lines:
                print(f"  {line}")
    except subprocess.TimeoutExpired:
        print("Timeout (>5 min) - generation interrompue")
    except Exception as e:
        print(f"Erreur : {type(e).__name__} - {str(e)[:150]}")

elif test_songgen and not songgen_ready:
    print("SongGeneration 2 non installe - voir Section 3 pour les instructions")
    print(f"\nExemple de commande CLI :")
    print(f"  bash generate.sh songgeneration_model input.jsonl output/ --separate")
    print(f"\nContenu input.jsonl :")
    songgen_input = {
        "idx": "morning_light",
        "gt_lyric": songgen_lyrics,
        "descriptions": test_song["genre"],
    }
    print(f"  {json.dumps(songgen_input, indent=2)}")
elif not test_songgen:
    print("Test SongGeneration 2 desactive")
else:
    print("GPU non disponible")

GENERATION AVEC SONGGENERATION 2
SongGeneration 2 non installe - voir Section 3 pour les instructions

Exemple de commande CLI :
  bash generate.sh songgeneration_model input.jsonl output/ --separate

Contenu input.jsonl :
  {
  "idx": "morning_light",
  "gt_lyric": "[intro-short] ; [verse] Walking through the morning light. Coffee steam and city sights. Every step a brand new start. Rhythm beating in my heart. ; [chorus] Sing it loud sing it free. This is where I want to be. Every note a part of me. Music sets my spirit free. ; [outro-short]",
  "descriptions": "pop, uplifting, piano, acoustic guitar, female vocal"
}


### Interpretation : SongGeneration 2

| Aspect | Observation typique |
|--------|--------------------|
| Qualite vocale | Proche de la qualite commerciale |
| Fidelite paroles | Excellente (PER 8.55%, meilleur que Suno v5) |
| Musicalite | Production elevee, mix professionnel |
| Temps | ~25s pour 30s audio (RTF 0.82) |

**Points cles** :
1. Beaucoup plus rapide que YuE (10-15x)
2. La separation vocale/accompagnement est native
3. Sortie en FLAC 48kHz (meilleure qualite que le MP3 de YuE)
4. Les descriptions doivent etre des tags, pas des phrases

## Section 7 : Comparaison et evaluation

Grille d'evaluation pour comparer les deux modeles sur les memes paroles.

In [9]:
# Tableau comparatif
print("COMPARAISON YUE vs SONGGENERATION 2")
print("=" * 55)

comparison = {
    "Critere": [
        "Architecture",
        "Parametres",
        "VRAM minimum",
        "VRAM recommandee",
        "Vitesse (30s audio)",
        "Format sortie",
        "Sample rate",
        "Duree max",
        "Separation pistes",
        "Style transfer",
        "Langues",
        "Licence",
        "Quantization",
        "API Python",
    ],
    "YuE": [
        "Autoregressive 2 stages",
        "7B + 1B",
        "~6 GB (quantise)",
        "24 GB (bf16)",
        "~6 min (RTX 4090)",
        "MP3",
        "44.1 kHz",
        "Illimitee",
        "Oui (vocal + instru)",
        "Oui (ICL, 30s ref)",
        "EN, ZH, JA, KO",
        "Apache 2.0",
        "exllamav2 (4.25bpw)",
        "CLI uniquement",
    ],
    "SongGeneration 2": [
        "Hybride LLM + Diffusion",
        "4B",
        "10 GB (base)",
        "22 GB (v2-large)",
        "~25s (H20 GPU)",
        "FLAC",
        "48 kHz",
        "4m30",
        "Oui (--separate)",
        "Oui (10s ref audio)",
        "EN, ZH, ES, JA",
        "Custom",
        "Non disponible",
        "CLI uniquement",
    ],
}

# Afficher le tableau
print(f"{'Critere':<25} {'YuE':<30} {'SongGeneration 2':<30}")
print("-" * 85)
for i, critere in enumerate(comparison["Critere"]):
    yue_val = comparison["YuE"][i]
    sg_val = comparison["SongGeneration 2"][i]
    print(f"{critere:<25} {yue_val:<30} {sg_val:<30}")

# Resultats de generation (si disponibles)
if yue_result or songgen_result:
    print(f"\n\nRESULTATS DE GENERATION")
    print("=" * 55)
    print(f"{'Metrique':<25} {'YuE':<30} {'SongGeneration 2':<30}")
    print("-" * 85)

    yue_dur = f"{yue_result['duration']:.1f}s" if yue_result else "N/A"
    sg_dur = f"{songgen_result['duration']:.1f}s" if songgen_result else "N/A"
    print(f"{'Duree':<25} {yue_dur:<30} {sg_dur:<30}")

    yue_time = f"{yue_result['gen_time']:.1f}s" if yue_result else "N/A"
    sg_time = f"{songgen_result['gen_time']:.1f}s" if songgen_result else "N/A"
    print(f"{'Temps generation':<25} {yue_time:<30} {sg_time:<30}")

    yue_rtf = f"{yue_result['rtf']:.2f}x" if yue_result else "N/A"
    sg_rtf = f"{songgen_result['rtf']:.2f}x" if songgen_result else "N/A"
    print(f"{'RTF':<25} {yue_rtf:<30} {sg_rtf:<30}")

    yue_sr = f"{yue_result['sample_rate']} Hz" if yue_result else "N/A"
    sg_sr = f"{songgen_result['sample_rate']} Hz" if songgen_result else "N/A"
    print(f"{'Sample rate':<25} {yue_sr:<30} {sg_sr:<30}")
else:
    print(f"\nAucun modele execute - tableau base sur les specifications")

COMPARAISON YUE vs SONGGENERATION 2
Critere                   YuE                            SongGeneration 2              
-------------------------------------------------------------------------------------
Architecture              Autoregressive 2 stages        Hybride LLM + Diffusion       
Parametres                7B + 1B                        4B                            
VRAM minimum              ~6 GB (quantise)               10 GB (base)                  
VRAM recommandee          24 GB (bf16)                   22 GB (v2-large)              
Vitesse (30s audio)       ~6 min (RTX 4090)              ~25s (H20 GPU)                
Format sortie             MP3                            FLAC                          
Sample rate               44.1 kHz                       48 kHz                        
Duree max                 Illimitee                      4m30                          
Separation pistes         Oui (vocal + instru)           Oui (--separate)             

### Interpretation : Comparaison

| Critere | Gagnant | Raison |
|---------|---------|--------|
| Qualite audio | SongGen 2 | 48kHz FLAC, mix professionnel |
| Vitesse | SongGen 2 | 10-15x plus rapide |
| Fidelite paroles | SongGen 2 | PER 8.55% (meilleur que Suno v5) |
| Accessibilite GPU | YuE | Quantization jusqu'a 6 GB |
| Duree max | YuE | Illimitee vs 4m30 |
| Licence | YuE | Apache 2.0 vs licence custom |
| Communaute | YuE | Plus d'outils tiers (exllamav2, Gradio) |

### Recommandations par cas d'usage

| Besoin | Recommandation |
|--------|----------------|
| Meilleure qualite possible | SongGeneration 2 (v2-large) |
| GPU limite (<12 GB) | YuE quantise (exllamav2) |
| Chansons longues (>5 min) | YuE (pas de limite de duree) |
| Production rapide | SongGeneration 2 (quasi temps reel) |
| Usage commercial | YuE (Apache 2.0) |
| Style transfer | Les deux (ICL/audio prompt) |
| Education / recherche | Les deux |

In [10]:
# Mode interactif
if notebook_mode == "interactive" and not skip_widgets:
    print("MODE INTERACTIF")
    print("=" * 45)
    print("\nEcrivez vos propres paroles (format simplifie) :")
    print("  Entrez un couplet et un refrain, separes par une ligne vide")
    print("  (Laissez vide pour passer)")

    try:
        user_input = input("\nCouplet (1 ligne) : ").strip()
        if user_input:
            user_chorus = input("Refrain (1 ligne) : ").strip()
            user_genre = input("Genre (tags) [pop, female] : ").strip() or "pop, female"

            custom_song = {
                "title": "Custom",
                "genre": user_genre,
                "sections": [
                    ("verse", [user_input]),
                    ("chorus", [user_chorus] if user_chorus else [user_input]),
                ],
            }

            print(f"\nFormat YuE :")
            print(format_lyrics_yue(custom_song))
            print(f"\nFormat SongGeneration 2 :")
            print(format_lyrics_songgen(custom_song))
            print(f"\nPour generer, sauvegardez ces fichiers et executez les commandes CLI")
        else:
            print("Mode interactif ignore")

    except (KeyboardInterrupt, EOFError):
        print("Mode interactif interrompu")
    except Exception as e:
        error_type = type(e).__name__
        if "StdinNotImplemented" in error_type or "input" in str(e).lower():
            print("Mode interactif non disponible (execution automatisee)")
        else:
            print(f"Erreur : {error_type} - {str(e)[:100]}")
else:
    print("Mode batch - Interface interactive desactivee")

MODE INTERACTIF

Ecrivez vos propres paroles (format simplifie) :
  Entrez un couplet et un refrain, separes par une ligne vide
  (Laissez vide pour passer)
Mode interactif non disponible (execution automatisee)


La section interactive permet de saisir ses propres paroles pour la generation. La cellule suivante compile les statistiques de session et libere les ressources GPU occupees par les modeles de generation musicale.

In [11]:
# Statistiques de session
print("STATISTIQUES DE SESSION")
print("=" * 45)

print(f"Date : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Device : {device}")
print(f"YuE : {'execute' if yue_result else 'non execute'}")
print(f"SongGeneration 2 : {'execute' if songgen_result else 'non execute'}")

if gpu_available:
    vram_current = torch.cuda.memory_allocated(0) / (1024**3)
    print(f"VRAM utilisee : {vram_current:.2f} GB")

if save_results:
    all_audio = list(OUTPUT_DIR.rglob('*.flac')) + list(OUTPUT_DIR.rglob('*.mp3')) + \
                list(OUTPUT_DIR.rglob('*.wav'))
    total_size = sum(f.stat().st_size for f in all_audio) / (1024*1024)
    print(f"\nFichiers audio : {len(all_audio)} ({total_size:.1f} MB)")
    print(f"Repertoire : {OUTPUT_DIR}")

# Liberation memoire
gc.collect()
if gpu_available:
    torch.cuda.empty_cache()

print(f"\nPROCHAINES ETAPES")
print(f"1. Comparer avec la generation instrumentale MusicGen (02-3)")
print(f"2. Comparer avec la generation MIDI symbolique (02-6)")
print(f"3. Utiliser la separation de pistes avec Demucs (02-4)")
print(f"4. Explorer le style transfer avec un audio de reference")

print(f"\nNotebook Generation de Chansons termine - {datetime.now().strftime('%H:%M:%S')}")

STATISTIQUES DE SESSION
Date : 2026-03-22 13:35:11
Device : cpu
YuE : non execute
SongGeneration 2 : non execute

Fichiers audio : 0 (0.0 MB)
Repertoire : MyIA.AI.Notebooks\GenAI\outputs\audio\songs



PROCHAINES ETAPES
1. Comparer avec la generation instrumentale MusicGen (02-3)
2. Comparer avec la generation MIDI symbolique (02-6)
3. Utiliser la separation de pistes avec Demucs (02-4)
4. Explorer le style transfer avec un audio de reference

Notebook Generation de Chansons termine - 13:35:11


---

# EXERCICE - Chanson personnalisee

## Objectif

Generer une chanson courte avec l'un des deux modeles et evaluer le resultat.

## Consignes

1. **Ecrire des paroles originales** :
   - 1 couplet (4-6 lignes)
   - 1 refrain (2-4 lignes)
   - Theme libre

2. **Choisir un style** :
   - Definir 3-5 tags de description (genre, instruments, voix, emotion)
   - Justifier le choix par rapport aux paroles

3. **Formater les paroles** :
   - Creer le fichier d'entree pour le modele choisi (YuE ou SongGen2)
   - Respecter le format de marqueurs de section

4. **Generer et evaluer** :
   - Generer la chanson
   - Evaluer sur 3 criteres (1-5) : fidelite paroles, qualite vocale, musicalite
   - Identifier les forces et faiblesses du resultat

## Indices

- Utilisez `format_lyrics_yue()` ou `format_lyrics_songgen()` pour formater vos paroles
- SongGeneration 2 base est le plus accessible (10 GB VRAM)
- Evitez les paroles trop longues (4-6 lignes par section max)
- Les tags en anglais donnent de meilleurs resultats

## Criteres de succes

- [ ] Paroles originales avec structure couplet/refrain
- [ ] Fichier d'entree correctement formate
- [ ] Chanson generee (ou tentative documentee si echec technique)
- [ ] Evaluation avec scores et justification

---

**Soumission** : PR avec titre "Exercice Song - [Votre Nom]", paroles, fichier audio et evaluation